In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  
import torch


device = torch.device("cuda:0")
torch.cuda.set_device(device)

In [ ]:
import pyterrier as pt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

/tmp/ipykernel_625583/986627200.py:9: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [3]:
# Step 1: Load dataset
dataset = pt.get_dataset("msmarco_passage")
topics19 = dataset.get_topics("test-2019")
qrels2019 = dataset.get_qrels("test-2019")
topics2019 = topics19.merge(qrels2019, on = "qid")[["qid","query"]].drop_duplicates()
topics20 = dataset.get_topics("test-2020")
qrels2020 = dataset.get_qrels("test-2020")
topics2020 = topics20.merge(qrels2020, on = "qid")[["qid","query"]].drop_duplicates()
topics_dlhard = pt.get_dataset('irds:msmarco-passage/trec-dl-hard').get_topics()
qrels_dlhard = pt.get_dataset('irds:msmarco-passage/trec-dl-hard').get_qrels()

In [ ]:
from pyterrier_colbert.ranking import ColBERTv2Index
colbert_model_path = '/data2/wangxiao/pyterrier_colbert2/colbertv2.0'
factory = ColBERTv2Index(colbert=colbert_model_path, 
                         index_location="/data2/wangxiao/pyterrier_colbert2/indices/msmarco_psg_v1_index/2bits/indexes/2bits/",
                           plaid_mode=True, ncells=4, centroid_score_threshold=0.4, ndocs=4096) 

[Jan 29, 12:39:50] #> Loading codec...
[Jan 29, 12:39:50] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/data2/wangxiao/anaconda3/envs/v2/lib/python3.8/site-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


[Jan 29, 12:39:50] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/data2/wangxiao/anaconda3/envs/v2/lib/python3.8/site-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


[Jan 29, 12:39:50] #> Loading IVF...
[Jan 29, 12:39:51] #> Loading doclens...


100%|███████████████████████████████████████| 354/354 [00:00<00:00, 1229.14it/s]


[Jan 29, 12:39:53] #> Loading codes and residuals...


100%|█████████████████████████████████████████| 354/354 [00:17<00:00, 20.32it/s]

 V2 retrieval time: 121.59 seconds


In [6]:
from plaid_prf_tools import *

# Effectiveness of PLAID-PRF

In [ ]:
qe_tfidf = plaid_prf(
    factory, dataset,
    top_psg=3, top_exp=14, beta=0.7,
    lambda_div=0.3,                
    output_exptok = False,
    weighting='tf-idf'
)

In [ ]:
from pyterrier.measures import AP, nDCG, RR, R
baseline = factory.end_to_end(k=1000)
pd.set_option('display.float_format', '{:.4f}'.format)
res = pt.Experiment(
    [ baseline,
     qe_tfidf>> plaid_end_to_end_qe(factory, k=1000), 
    
    ],
    topics2019, qrels2019,
    eval_metrics=[ AP(rel=2), nDCG@10,  RR(rel=2)@10,
                   R(rel=2)@1000,],
    names=['PLAID','PALID-PRF', ],
    verbose=True,  baseline=0)
res

pt.Experiment: 100%|██████████████████████████| 2/2 [00:06<00:00,  3.30s/system]


,name,nDCG@10,AP(rel=2),R(rel=2)@1000,RR(rel=2)@10,nDCG@10 +,nDCG@10 -,nDCG@10 p-value,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value,R(rel=2)@1000 +,R(rel=2)@1000 -,R(rel=2)@1000 p-value,RR(rel=2)@10 +,RR(rel=2)@10 -,RR(rel=2)@10 p-value
0,PLAID,0.7383,0.5097,0.8706,0.8934,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PALID-PRF,0.7700,0.5282,0.8967,0.9287,23.0000,11.0000,0.0297,26.0000,16.0000,0.1358,10.0000,9.0000,0.1206,5.0000,2.0000,0.2478


# Efficiency of PLAID-PRF

In [16]:
from plaid_prf_latency import *

In [14]:
initial_stage = make_initial_stage(
    factory,
    dataset,
    top_psg=3,          # 你要用的 PRF depth
    output_exptok=False  # 如果不需要 wpids，可以设为 False
)

In [21]:
prof_before = plaid_end_to_end_qe_profile_report(factory, k=3, materialize =True)
prof_after = plaid_end_to_end_qe_profile_report(factory, k=1000, materialize =True)

# BEFORE
before_lat = latency_profile_n_runs(
    prof_before, topics2019,
    n_runs=5, warmup=1,
    agg_over_queries="median",
    agg_over_runs="median"
)

# AFTER：先跑 PRF 拿 df_qe（含 query_vec）
df_stage1 = initial_stage(topics2019)

prf_stage = make_prf_stage_from_stage1_new(
    idf_map=idf_map,
    N_global=stats['N'],
    eps=stats['eps'],
    add_one=stats['add_one'],
    top_psg= 3,
    top_exp=14,
    beta=1,
    lambda_div=0.3,
    lambda_q=0.3,
    mmr_selection=True,
    weighting="tf-idf",
    output_exptok=False,  
)
df_qe = prf_stage(df_stage1)

after_lat = latency_profile_n_runs(
    prof_after, df_qe,
    n_runs=5, warmup=1,
    agg_over_queries="median",
    agg_over_runs="median"
)

print("before:\n", before_lat)
print("after:\n", after_lat)


before:
 lat_query_encoding_ms         14.0390
lat_candidate_generation_ms    1.8909
lat_score_pids_total_ms       28.1768
lat_materialize_ms             0.7347
lat_total_ms                  45.2715
dtype: float64
after:
 lat_query_encoding_ms          0.0903
lat_candidate_generation_ms    2.2537
lat_score_pids_total_ms       34.8797
lat_materialize_ms            10.5072
lat_total_ms                  47.6815
dtype: float64
